In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from config import SimConfig

In [ ]:
# --- 1. PHYSICAL ANCHORS (LOCKED CONSTANTS) ---
cfg = SimConfig()

A0 = cfg.a0            # g/s 
A1 = cfg.a1            # g/(s kW)
A2 = cfg.a2            # g/(s kW^2)
P_MAX = cfg.p_max     # kW

K_FC = 750.*P_MAX      # Euro
LHV = 33.33            # kWh/kg

EPS = 1e-9

K_E = (3600 * LHV) / 1000

P_D_MAX = 2400


In [ ]:
# ==========================================
# 1. UI WIDGET INITIALIZATION
# ==========================================
style = {'description_width': 'initial'}
center_layout = widgets.Layout(justify_content='center')

# Sliders (Continuous update enabled for real-time WebGL rendering!)
w1_p_nom = widgets.FloatSlider(value=80.0, min=60.0, max=P_MAX, step=1.0, description='p_nom (kW):', style=style)
w2_p_opt = widgets.FloatSlider(value=70.0, min=32.0, max=P_MAX, step=1.0, description='p_opt (kW):', style=style)
w2_eta_opt = widgets.FloatSlider(value=0.55, min=0.45, max=0.60, step=0.005, description='\u03B7_opt:', style=style, readout_format='.3f')
w2_tgt_kh2 = widgets.FloatSlider(value=4.0, min=1.6, max=16.0, step=0.1, description='Target H2 (€/kg):', style=style)
w2_tgt_tau = widgets.IntSlider(value=80000, min=10000, max=100000, step=2000, description='Target Life (h):', style=style)
w2_tgt_smax = widgets.IntSlider(value=4000, min=1500, max=15000, step=500, description='Target S_max:', style=style)

w3_lambda1 = widgets.FloatLogSlider(value=1e-4, base=10, min=-7, max=-2, step=0.01, description='λ₁ (OPEX):', style=style, readout_format='.2e')
w3_lambda2 = widgets.FloatLogSlider(value=1e4, base=10, min=2, max=7, step=0.01, description='λ₂ (Stubbornness):', style=style, readout_format='.2e')

# Native Widget Containers
box_l1 = widgets.HBox([], layout=center_layout)
box_l2 = widgets.HBox([], layout=center_layout)
box_l3 = widgets.HBox([], layout=center_layout)

# ==========================================
# 2. STATEFUL FIGURE INITIALIZATION
# ==========================================
# --- Figure 1 ---
fig1 = go.FigureWidget()
fig1.add_trace(go.Heatmap(colorscale='Viridis', zmin=0, zmax=3.0, colorbar=dict(title=r'Degradation (α)', xpad=10), hovertemplate='p_opt: %{x:.1f} kW<br>eta_opt: %{y:.3f}<br>Alpha: %{z:.2f}<extra></extra>'))
fig1.update_layout(height=750, width=1350, plot_bgcolor='rgba(230, 230, 230, 1)', margin=dict(t=120))

# --- Figure 2 (16 Pre-allocated Traces) ---
fig2 = go.FigureWidget(make_subplots(rows=2, cols=2, subplot_titles=("1. Hydrogen Price", "2. Stack Lifetime", "3. Mechanical Limit (S_max)", "4. Heuristic Stubbornness (k_s)"), vertical_spacing=0.15, horizontal_spacing=0.18))
fig2.add_trace(go.Scatter(line=dict(color='black', width=3)), row=1, col=1) # 0: k_h2 line
fig2.add_trace(go.Scatter(line=dict(color='black', width=3)), row=1, col=2) # 1: tau_fc line

# FIX: Localized the colorbars to their specific subplots using len, x, and y parameters
fig2.add_trace(go.Heatmap(colorscale='Viridis', zmin=1000, zmax=20000, colorbar=dict(title="Switches", len=0.45, y=0.22, x=0.46)), row=2, col=1) # 2: S_MAX map
fig2.add_trace(go.Heatmap(colorscale='Plasma', zmax=100, colorbar=dict(title="€/Switch", len=0.45, y=0.22, x=1.02)), row=2, col=2) # 3: K_S map

fig2.add_trace(go.Scatter(mode='lines', line=dict(color='white', width=3, dash='dash')), row=2, col=1) # 4: S_max Diag 1
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='white', width=3, dash='dash')), row=2, col=2) # 5: S_max Diag 2
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='black', dash='dash')), row=1, col=1) # 6: Target H2 H-line
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='black', dash='dash')), row=1, col=2) # 7: Target Tau H-line

# 8-11: Green Fuel Target Lines (All 4 Subplots)
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#00CC96', width=3)), row=1, col=1) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#00CC96', width=3)), row=1, col=2) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#00CC96', width=3)), row=2, col=1) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#00CC96', width=3)), row=2, col=2) 

# 12-15: Red Life Target Lines (All 4 Subplots)
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#EF553B', width=3)), row=1, col=1) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#EF553B', width=3)), row=1, col=2) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#EF553B', width=3)), row=2, col=1) 
fig2.add_trace(go.Scatter(mode='lines', line=dict(color='#EF553B', width=3)), row=2, col=2) 

# FIX: Increased top margin to 230 to prevent multiline titles from overlapping subplot headers
fig2.update_layout(height=850, width=1350, showlegend=False, plot_bgcolor='rgba(245, 245, 245, 1)', margin=dict(t=230))
for r in [1, 2]:
    for c in [1, 2]: fig2.update_xaxes(type="log", title_text="λ₁ (OPEX Multiplier)", row=r, col=c)
fig2.update_yaxes(type="log", title_text="λ₂ (Stubbornness)", row=2, col=1)
fig2.update_yaxes(type="log", title_text="λ₂ (Stubbornness)", row=2, col=2)

# --- Figure 3 (3x1 Vertical Layout) ---
fig3 = go.FigureWidget(make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("1. Optimal Active Modules", "2. Cost Penalty vs Optimal (%)", "3. Break-Even Time [Hours]"), vertical_spacing=0.08))
fig3.add_trace(go.Scatter(name="Analytical", line=dict(color='gray', dash='dash', width=2)), row=1, col=1) # 0: Cont Opt
fig3.add_trace(go.Scatter(name="Discrete", line=dict(color='purple', width=2, shape='hv')), row=1, col=1) # 1: Disc Opt

# FIX: Localized the colorbars vertically using y and len.
fig3.add_trace(go.Heatmap(colorscale='Viridis', zmin=0, zmax=40, colorbar=dict(title="% Penalty", len=0.28, y=0.5), hovertemplate='P_d: %{x:.0f} kW<br>Active: %{y}<br>Penalty: %{z:.1f}%<extra></extra>'), row=2, col=1) # 2: Penalty Map
fig3.add_trace(go.Heatmap(colorscale='Plasma', zmin=0, zmax=8.0, colorbar=dict(title="Hours", len=0.28, y=0.14), hovertemplate='P_d: %{x:.0f} kW<br>Active: %{y}<br>Break-Even: %{z:.2f} hrs<extra></extra>'), row=3, col=1) # 3: Time Map

fig3.update_layout(height=1050, width=1350, plot_bgcolor='rgba(245, 245, 245, 1)', margin=dict(t=120), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
fig3.update_yaxes(title_text="Active Modules (n)", range=[0.5, 16.5], tickmode='linear', tick0=1, dtick=2)

# FIX: Force tick labels to show on all 3 subplots while keeping the single title at the bottom
fig3.update_xaxes(showticklabels=True)
fig3.update_xaxes(title_text="Power Demand P_d [kW]", row=3, col=1)

# ==========================================
# 3. OBSERVER LOGIC & BATCH UPDATING
# ==========================================

# FIX: Helper function to scrub NaNs to 'None' and explicitly convert Numpy Arrays to native lists.
def to_safe_list(arr):
    if not isinstance(arr, np.ndarray):
        arr = np.array(arr)
    arr_list = arr.tolist()
    if arr.ndim == 1:
        return [None if (isinstance(x, float) and np.isnan(x)) else x for x in arr_list]
    elif arr.ndim == 2:
        return [[None if (isinstance(x, float) and np.isnan(x)) else x for x in row] for row in arr_list]
    return arr_list

def master_observer(*args):
    p_nom = w1_p_nom.value
    p_opt = w2_p_opt.value
    eta_opt = w2_eta_opt.value
    t_kh2 = w2_tgt_kh2.value
    t_tau = w2_tgt_tau.value
    t_smax = w2_tgt_smax.value
    l1 = w3_lambda1.value
    l2 = w3_lambda2.value
    
    # 1. Enforce Slider Bounds
    raw_floor = (A0 / A2)**0.5
    p_opt_floor = np.ceil(raw_floor * 1) / 1 
    p_opt_ceil = np.floor((p_nom - 1) * 1) / 1
    if p_opt_ceil > w2_p_opt.max: w2_p_opt.max = p_opt_ceil
    w2_p_opt.min, w2_p_opt.max = p_opt_floor, p_opt_ceil
    p_opt = w2_p_opt.value 
    
    d1 = K_E * (A1 + 2 * A2 * p_opt)
    eta_floor = max(0.20, min(1.0 / d1 if d1 != 0 else 0.4, 0.90))
    d2 = K_E * ((A1 + 2 * A2 * p_nom) * (p_nom + p_opt) - 2 * (A2 * p_nom**2 - A0))
    eta_ceil = max(eta_floor + 0.01, min((p_nom + p_opt) / d2 if d2 > 0 else 0.8, 0.95))
    if eta_ceil > w2_eta_opt.max: w2_eta_opt.max = eta_ceil
    w2_eta_opt.min, w2_eta_opt.max = np.ceil(eta_floor*100)/100, np.floor(eta_ceil*100)/100
    eta_opt = w2_eta_opt.value 
    
    # 2. Global Math
    K_eff = 3600 * LHV * eta_opt
    D = A1 + 2 * p_nom * A2
    denom = K_eff * D - 1000
    
    invalid_physics = False
    if abs(denom) < 1e-9: invalid_physics = True
    else:
        M = (2000 * (p_nom - p_opt)) / denom
        X_const = (M * K_eff) / 1000
        Y_const = (p_opt**2) - (p_nom**2) - X_const * (A0 - A2 * p_nom**2)
        num = (1 - A2 * X_const) * p_nom**2
        if Y_const <= 0 or num <= 0 or M <= 0: invalid_physics = True
        else: alpha = num / Y_const

    # --- UPDATE FIGURE 1 ---
    with fig1.batch_update():
        if invalid_physics:
            fig1.data[0].z = None
            fig1.layout.title.text = f"<span style='color:red; font-size:24px'><b>PHYSICS VIOLATION: INVALID CONFIGURATION</b></span>"
        else:
            p_opt_range = np.linspace(p_opt_floor + 0.1, p_nom - 0.1, 250)
            eta_floor_c = 1.0 / (K_E * (A1 + 2 * A2 * p_opt_range))
            eta_ceil_c = (p_nom + p_opt_range) / (K_E * ((A1 + 2 * A2 * p_nom) * (p_nom + p_opt_range) - 2 * (A2 * p_nom**2 - A0)))
            eta_opt_range = np.linspace(np.nanmin(eta_floor_c), np.nanmax(eta_ceil_c), 250)
            P_OPT, ETA_OPT = np.meshgrid(p_opt_range, eta_opt_range)
            
            K_eff_grid = 3600 * LHV * ETA_OPT
            M_grid = (2000 * (p_nom - P_OPT)) / (K_eff_grid * D - 1000)
            X_grid = (M_grid * K_eff_grid) / 1000
            num_grid = (1 - A2 * X_grid) * (p_nom**2)
            den_grid = (P_OPT**2) - (p_nom**2) - X_grid * (A0 - A2 * p_nom**2)
            ALPHA_grid = np.where((den_grid > 0) & (num_grid > 0), num_grid / den_grid, np.nan)
            
            # FIX: Used to_safe_list everywhere to prevent JSON and ValueError crashes
            fig1.data[0].z = to_safe_list(ALPHA_grid)
            fig1.data[0].x = to_safe_list(p_opt_range)
            fig1.data[0].y = to_safe_list(eta_opt_range)
            fig1.layout.title.text = (
                f"<span style='font-size:20px'><b>Level 1: Thermodynamic Feasibility Envelope</b></span><br>"
                f"<span style='font-size:14px; color:#444'>Nominal Cruising Baseline: <b>p_nom = {p_nom} kW</b></span>"
            )

    # --- UPDATE FIGURE 2 & 3 ---
    with fig2.batch_update(), fig3.batch_update():
        if invalid_physics:
            for t in fig2.data: t.x, t.y, t.z = None, None, None
            for t in fig3.data: t.x, t.y, t.z = None, None, None
            err_msg = f"<span style='color:red; font-size:20px'><b>PHYSICS VIOLATION</b></span>"
            fig2.layout.title.text, fig3.layout.title.text = err_msg, err_msg
        else:
            # Level 2 Variables
            lambda1_k = t_kh2 / (M * K_eff)
            lambda1_tau = K_FC / (3600 * Y_const * t_tau)
            lambda2_k = K_FC / (lambda1_k * t_smax)
            lambda2_tau = K_FC / (lambda1_tau * t_smax)

            l1_min, l1_max = min(lambda1_k, lambda1_tau), max(lambda1_k, lambda1_tau)
            l1_range = np.logspace(np.log10(l1_min) - 1, np.log10(l1_max) + 1, 200)
            l1_cen = 10**((np.log10(l1_min) + np.log10(l1_max)) / 2)
            l2_cen = K_FC / (l1_cen * t_smax)
            l2_range = np.logspace(np.log10(l2_cen) - 1.5, np.log10(l2_cen) + 1.5, 200)

            # Bind Target Sliders dynamically
            l1_min_exp, l1_max_exp = np.floor(np.log10(l1_min)) - 1.0, np.ceil(np.log10(l1_max)) + 1.0
            if l1_max_exp > w3_lambda1.max: w3_lambda1.max = l1_max_exp
            w3_lambda1.min, w3_lambda1.max = l1_min_exp, l1_max_exp
            l2_exp = np.log10(l2_cen)
            if l2_exp + 2.0 > w3_lambda2.max: w3_lambda2.max = l2_exp + 2.0
            w3_lambda2.min, w3_lambda2.max = l2_exp - 2.0, l2_exp + 2.0

            # Map Tab 2 Plots (Safe Lists)
            fig2.data[0].x, fig2.data[0].y = to_safe_list(l1_range), to_safe_list((M * K_eff) * l1_range)
            fig2.data[1].x, fig2.data[1].y = to_safe_list(l1_range), to_safe_list(K_FC / (3600 * Y_const * l1_range))
            L1, L2 = np.meshgrid(l1_range, l2_range)
            fig2.data[2].x, fig2.data[2].y, fig2.data[2].z = to_safe_list(l1_range), to_safe_list(l2_range), to_safe_list(K_FC / (L1 * L2))
            fig2.data[3].x, fig2.data[3].y, fig2.data[3].z = to_safe_list(l1_range), to_safe_list(l2_range), to_safe_list(L1 * L2)
            diag = K_FC / (l1_range * t_smax)
            fig2.data[4].x, fig2.data[4].y = to_safe_list(l1_range), to_safe_list(diag)
            fig2.data[5].x, fig2.data[5].y = to_safe_list(l1_range), to_safe_list(diag)
            
            # Map Tab 2 Intersections
            fig2.data[6].x, fig2.data[6].y = to_safe_list(l1_range), [t_kh2]*200
            fig2.data[7].x, fig2.data[7].y = to_safe_list(l1_range), [t_tau]*200
            
            y1_bounds = [0, t_kh2 * 2]
            y2_bounds = [0, t_tau * 2]
            y3_bounds = [10**(np.log10(l2_cen) - 1.5), 10**(np.log10(l2_cen) + 1.5)]
            
            # Green Lines (Fuel Target) on all 4 plots
            fig2.data[8].x, fig2.data[8].y = [lambda1_k, lambda1_k], y1_bounds
            fig2.data[9].x, fig2.data[9].y = [lambda1_k, lambda1_k], y2_bounds
            fig2.data[10].x, fig2.data[10].y = [lambda1_k, lambda1_k], y3_bounds
            fig2.data[11].x, fig2.data[11].y = [lambda1_k, lambda1_k], y3_bounds

            # Red Lines (Life Target) on all 4 plots
            fig2.data[12].x, fig2.data[12].y = [lambda1_tau, lambda1_tau], y1_bounds
            fig2.data[13].x, fig2.data[13].y = [lambda1_tau, lambda1_tau], y2_bounds
            fig2.data[14].x, fig2.data[14].y = [lambda1_tau, lambda1_tau], y3_bounds
            fig2.data[15].x, fig2.data[15].y = [lambda1_tau, lambda1_tau], y3_bounds

            fig2.layout.yaxis.range = y1_bounds
            fig2.layout.yaxis2.range = y2_bounds

            status_col = "#00CC96" if abs(lambda1_k - lambda1_tau) / lambda1_k < 0.05 else "black"
            msg = "🎯 TARGETS ALIGNED!" if status_col == "#00CC96" else "Adjust p_opt / η_opt to merge lines"
            
            fig2.layout.title.text = (
                f"<span style='font-size:20px'><b>Level 2: Hardware Target Solver</b></span><br>"
                f"<span style='font-size:14px; color:#444'>Hardware Anchors: <b>p_nom = {p_nom} kW</b> | Calculated Degradation: <b>α = {alpha:.3f}</b></span><br>"
                f"<b style='font-size:16px; color:{status_col}'>{msg}</b><br>"
                f"<span style='font-size:13px; color:#00CC96'>● Fuel Target (Green): λ₁ = <b>{lambda1_k:.2e}</b> ➔ Requires λ₂ = <b>{lambda2_k:.1e}</b></span> | "
                f"<span style='font-size:13px; color:#EF553B'>● Life Target (Red): λ₁ = <b>{lambda1_tau:.2e}</b> ➔ Requires λ₂ = <b>{lambda2_tau:.1e}</b></span>"
            )
            fig2.layout.title.x = 0.5
            fig2.layout.title.xanchor = 'center'

            # Level 3 Variables
            A, B, C = l1, (M - 2 * p_opt) * l1, (p_opt**2) * l1
            k_s = l1 * l2
            k_h2_val = M * K_eff * l1
            tau_fc_val = K_FC / (3600 * Y_const * l1)
            s_max_val = K_FC / k_s if k_s > 0 else np.nan

            P_d = np.linspace(200, P_D_MAX+1, 100) # Capped at 2400 as requested
            n_array = np.arange(1, 17)
            P_D, N_GRID = np.meshgrid(P_d, n_array)
            
            C_o = A * (P_D**2 / N_GRID) + B * P_D + N_GRID * C
            C_opt_1D = np.min(C_o, axis=0)
            n_opt_disc = n_array[np.argmin(C_o, axis=0)]
            C_opt_2D = np.tile(C_opt_1D, (len(n_array), 1))
            n_opt_2D = np.tile(n_opt_disc, (len(n_array), 1))
            
            delta_C_percent = 100 * (C_o - C_opt_2D) / C_opt_2D
            savings_per_sec = C_o - C_opt_2D
            switch_cost_total = k_s * np.abs(N_GRID - n_opt_2D)
            
            # Map Break Even to HOURS
            with np.errstate(divide='ignore', invalid='ignore'):
                t_be_hours = np.where(savings_per_sec > 1e-6, (switch_cost_total / savings_per_sec) / 3600.0, np.nan)

            # Map Tab 3 Plots (Safe Lists)
            fig3.data[0].x, fig3.data[0].y = to_safe_list(P_d), to_safe_list(P_d / p_opt)
            fig3.data[1].x, fig3.data[1].y = to_safe_list(P_d), to_safe_list(n_opt_disc)
            fig3.data[2].x, fig3.data[2].y, fig3.data[2].z = to_safe_list(P_d), to_safe_list(n_array), to_safe_list(delta_C_percent)
            fig3.data[3].x, fig3.data[3].y, fig3.data[3].z = to_safe_list(P_d), to_safe_list(n_array), to_safe_list(t_be_hours)
            
            fig3.layout.title.text = (
                f"<span style='font-size:20px'><b>Level 3: Heuristic Controller Behavior</b></span><br>"
                f"<span style='font-size:14px; color:#444'>Calculated Degradation: <b>α = {alpha:.3f}</b> | "
                f"k_h2 = <b>{k_h2_val:.2f} €/kg</b> | tau_fc = <b>{tau_fc_val:,.0f} h</b> | k_s = <b>{k_s:.2f} €</b> | S_max = <b>{s_max_val:,.0f}</b></span>"
            )
            fig3.layout.title.x = 0.5
            fig3.layout.title.xanchor = 'center'

# Bind observer to all widgets
for w in [w1_p_nom, w2_p_opt, w2_eta_opt, w2_tgt_kh2, w2_tgt_tau, w2_tgt_smax, w3_lambda1, w3_lambda2]:
    w.observe(master_observer, 'value')

# ==========================================
# 4. CONSTRUCT LAYOUT
# ==========================================
# Inject Native Widgets into Containers
box_l1.children = [fig1]
box_l2.children = [fig2]
box_l3.children = [fig3]

controls_l1 = widgets.HBox([w1_p_nom], layout=center_layout)
tab1 = widgets.VBox([box_l1, controls_l1])

target_row = widgets.HBox([w2_tgt_kh2, w2_tgt_tau, w2_tgt_smax], layout=center_layout)
thermo_row = widgets.HBox([w2_p_opt, w2_eta_opt], layout=center_layout)
controls_l2 = widgets.VBox([
    widgets.HTML("<div style='text-align:center'><b>1. Define Hardware Targets (OPEX & CAPEX)</b></div>"), target_row,
    widgets.HTML("<div style='text-align:center'><b>2. Steer Thermodynamics to Align Targets</b></div>"), thermo_row
])
tab2 = widgets.VBox([box_l2, controls_l2])

inputs_l3 = widgets.HBox([w3_lambda1, w3_lambda2], layout=center_layout)
controls_l3 = widgets.VBox([
    widgets.HTML("<div style='text-align:center; padding-top:10px;'><b>Tune Controller (Notice how changing λ₁ scales prices but does not affect logic!)</b></div>"), inputs_l3
])
tab3 = widgets.VBox([box_l3, controls_l3])

dashboard = widgets.Tab(children=[tab1, tab2, tab3])
dashboard.set_title(0, 'Level 1: Thermodynamics')
dashboard.set_title(1, 'Level 2: Target Solver')
dashboard.set_title(2, 'Level 3: Heuristics Brain')

# Initialize plots
master_observer()
display(dashboard)